# Tutorial 1 — AI / semantic retrieval via MCP

Build a small document corpus with embeddings, index it with HNSW, and run the
six-step agent workflow that the BenoStreamDB **MCP server** exposes as tools:

1. discover a table
2. inspect its schema
3. execute SQL
4. perform vector retrieval
5. traverse an edge table
6. combine the results

Everything runs in-process against the embedded engine. The same SQL is
reachable from Spark, Trino, dbt, and Flight SQL — see
`docs/tutorials/01_ai_retrieval_mcp.md`.

In [ ]:
import shutil
import pyarrow as pa
import benostreamdb as bsdb

# Start clean so the notebook is re-runnable.
shutil.rmtree("/tmp/bsdb_tutorial1", ignore_errors=True)

# Pick a device. "cpu" is deterministic for a tutorial; use "auto" or
# "cuda:0" to run on a GPU.
print("GPU backend:", bsdb.set_gpu_device("cpu"))

docs = pa.table({
    "id": [1, 2, 3, 4],
    "title": ["Iceberg overlays", "Vector search", "Graph on the lakehouse", "JSON paths"],
    "embedding": [
        [1.0, 0.0, 0.0, 0.0],
        [0.9, 0.1, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
    ],
})
table = bsdb.Table.from_arrow("file:///tmp/bsdb_tutorial1/docs", docs)
table.add_index("embedding", "hnsw")
table.to_pandas()

## Step 4 — vector retrieval

`vector_search` uses the HNSW overlay; the query vector is the first document's
embedding, so the top hit should be document 1.

In [ ]:
# `vector_search` returns a pandas DataFrame directly.
hits = table.vector_search("embedding", [1.0, 0.0, 0.0, 0.0], k=2)
hits

## Steps 1–3, 5–6 — the agent workflow

The MCP server exposes these as tools (`list_graph_tables`, `execute_sql`,
`subscribe_events`, …). Here we drive the same primitives directly.

In [ ]:
session = bsdb.Session()
session.register("docs", table)

# 1. discover graph tables (the MCP `list_graph_tables` tool)
print("graph tables:", session.list_graph_tables())

# 2. inspect schema (the MCP `resources/read` on a table)
print(session.sql("DESCRIBE docs"))

# 3. execute SQL
print(session.sql("SELECT id, title FROM docs WHERE id <= 2"))

# 6. combine: SQL filter + vector ranking (see Tutorial 3 for step 5)
print(session.sql("SELECT id, title FROM docs ORDER BY id"))

## Wiring the MCP server into an agent

Build the server and point your MCP client (Claude Desktop, Cursor,
Antigravity) at it:

```bash
cargo build --release --manifest-path contrib/benostreamdb-mcp/Cargo.toml
```

```json
{
  "mcpServers": {
    "benostreamdb": {
      "command": "/path/to/benostreamdb-mcp",
      "env": { "BSDB_WAREHOUSE": "/tmp/bsdb_warehouse", "BSDB_GPU_DEVICE": "auto" }
    }
  }
}
```

Example tool calls the agent makes:

```json
{ "name": "list_graph_tables", "arguments": {} }
{ "name": "execute_sql", "arguments": { "query": "SELECT id, title FROM docs ORDER BY id" } }
{ "name": "subscribe_events", "arguments": { "table": "docs", "max_events": 5, "timeout_ms": 100 } }
```